<a href="https://www.kaggle.com/code/lokeswar56/seeing-through-sound?scriptVersionId=224036748" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
!pip install opencv-python Pillow transformers gtts ffmpeg-python google-cloud-texttospeech

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.7/186.7 kB 6.5 MB/s eta 0:00:00


In [3]:
cd /kaggle/input/parkvideo

/kaggle/input/parkvideo


In [4]:
ls

'videoplayback (online-video-cutter.com).mp4'


In [5]:
import cv2
import os

def extract_frames(video_path, output_folder, frame_rate=1):
    os.makedirs(output_folder, exist_ok=True)
    
    video = cv2.VideoCapture(video_path)
    fps = video.get(cv2.CAP_PROP_FPS)
    interval = int(fps * frame_rate)  # Capture every second

    count, saved = 0, 0
    while video.isOpened():
        ret, frame = video.read()
        if not ret:
            break
        if count % interval == 0:
            frame_path = os.path.join(output_folder, f"frame_{saved}.jpg")
            cv2.imwrite(frame_path, frame)
            saved += 1
        count += 1
    video.release()
    print(f"Extracted {saved} frames.")

extract_frames('videoplayback (online-video-cutter.com).mp4', '/kaggle/working/frames')


Extracted 299 frames.


In [6]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

# Load the model
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def generate_caption(image_path):
    image = Image.open(image_path).convert('RGB')
    inputs = processor(image, return_tensors="pt")
    output = model.generate(**inputs)
    caption = processor.decode(output[0], skip_special_tokens=True)
    return caption

# Generate captions for extracted frames
frame_folder = '/kaggle/working/frames'
captions = []
for frame in sorted(os.listdir(frame_folder)):
    frame_path = os.path.join(frame_folder, frame)
    caption = generate_caption(frame_path)
    captions.append(caption)
    print(f"{frame}: {caption}")


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

frame_0.jpg: a street with people walking around it
frame_1.jpg: a city street with people walking around it
frame_10.jpg: a crowd of people walking down a street
frame_100.jpg: a large machine that is making food
frame_101.jpg: a machine that is making food in a kitchen
frame_102.jpg: a large display case filled with lots of food
frame_103.jpg: a man is making pizzas in a kitchen
frame_104.jpg: a man is making pizzas in a restaurant
frame_105.jpg: a man is seen through a window in a building
frame_106.jpg: a man walking down a street in a city
frame_107.jpg: a man is walking down the street in a red shirt
frame_108.jpg: a man walking down a street in a city
frame_109.jpg: a group of people walking down a street
frame_11.jpg: a group of people walking down a street
frame_110.jpg: people walking down a street in the city
frame_111.jpg: people walking down the street in a city
frame_112.jpg: people walking down the street in a city
frame_113.jpg: people sitting on the sidewalk in front o

In [7]:
import cv2
import os
import numpy as np
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from gtts import gTTS

# Load the BLIP model for image captioning
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def extract_frames(video_path, output_folder, frame_rate=1):
    os.makedirs(output_folder, exist_ok=True)
    
    video = cv2.VideoCapture(video_path)
    fps = video.get(cv2.CAP_PROP_FPS)
    interval = int(fps * frame_rate)  # Capture every second

    count, saved = 0, 0
    prev_frame = None  # Store previous frame for movement detection

    while video.isOpened():
        ret, frame = video.read()
        if not ret:
            break

        if count % interval == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)  # Convert to grayscale for movement detection

            # Detect movement by comparing with previous frame
            if prev_frame is not None:
                diff = cv2.absdiff(prev_frame, gray)
                non_zero_count = np.count_nonzero(diff)
                movement_threshold = 5000  # Adjust threshold as needed
                
                if non_zero_count > movement_threshold:  # Movement detected
                    frame_path = os.path.join(output_folder, f"frame_{saved}.jpg")
                    cv2.imwrite(frame_path, frame)
                    saved += 1
            
            prev_frame = gray  # Update previous frame
        
        count += 1

    video.release()
    print(f"Extracted {saved} frames with movement.")

def generate_caption(image_path):
    image = Image.open(image_path).convert('RGB')
    inputs = processor(image, return_tensors="pt")
    output = model.generate(**inputs)
    caption = processor.decode(output[0], skip_special_tokens=True)
    return caption

def text_to_speech(text, output_path):
    tts = gTTS(text=text, lang="en")
    tts.save(output_path)

# Main Execution
video_path = 'videoplayback (online-video-cutter.com).mp4'
frame_folder = '/kaggle/working/frames'

# Step 1: Extract frames with movement detection
extract_frames(video_path, frame_folder)

# Step 2: Generate captions and convert to speech
for frame in sorted(os.listdir(frame_folder)):
    frame_path = os.path.join(frame_folder, frame)
    caption = generate_caption(frame_path)

    # Save audio file
    audio_path = os.path.join(frame_folder, f"{frame}.mp3")
    text_to_speech(caption, audio_path)
    
    print(f"{frame}: {caption} (Audio saved as {audio_path})")


Extracted 298 frames with movement.
frame_0.jpg: a city street with people walking around it (Audio saved as /kaggle/working/frames/frame_0.jpg.mp3)
frame_1.jpg: a large cathedral with many windows and people walking around (Audio saved as /kaggle/working/frames/frame_1.jpg.mp3)
frame_10.jpg: a group of people walking down a street (Audio saved as /kaggle/working/frames/frame_10.jpg.mp3)
frame_100.jpg: a machine that is making food in a kitchen (Audio saved as /kaggle/working/frames/frame_100.jpg.mp3)
frame_101.jpg: a large display case filled with lots of food (Audio saved as /kaggle/working/frames/frame_101.jpg.mp3)
frame_102.jpg: a man is making pizzas in a kitchen (Audio saved as /kaggle/working/frames/frame_102.jpg.mp3)
frame_103.jpg: a man is making pizzas in a restaurant (Audio saved as /kaggle/working/frames/frame_103.jpg.mp3)
frame_104.jpg: a man is seen through a window in a building (Audio saved as /kaggle/working/frames/frame_104.jpg.mp3)
frame_105.jpg: a man walking down a

# After uodating order frames

In [5]:
import cv2
import os
import numpy as np
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from gtts import gTTS

# Load the BLIP model for image captioning
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def extract_frames(video_path, output_folder, frame_rate=1):
    os.makedirs(output_folder, exist_ok=True)
    
    video = cv2.VideoCapture(video_path)
    fps = video.get(cv2.CAP_PROP_FPS)
    interval = int(fps * frame_rate)  # Capture every second

    count, saved = 0, 0
    prev_frame = None  # Store previous frame for movement detection

    while video.isOpened():
        ret, frame = video.read()
        if not ret:
            break

        if count % interval == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)  # Convert to grayscale for movement detection

            # Detect movement by comparing with previous frame
            if prev_frame is not None:
                diff = cv2.absdiff(prev_frame, gray)
                non_zero_count = np.count_nonzero(diff)
                movement_threshold = 5000  # Adjust threshold as needed
                
                if non_zero_count > movement_threshold:  # Movement detected
                    frame_path = os.path.join(output_folder, f"frame_{saved}.jpg")
                    cv2.imwrite(frame_path, frame)
                    saved += 1
            
            prev_frame = gray  # Update previous frame
        
        count += 1

    video.release()
    print(f"Extracted {saved} frames with movement.")

def generate_caption(image_path):
    image = Image.open(image_path).convert('RGB')
    inputs = processor(image, return_tensors="pt")
    output = model.generate(**inputs)
    caption = processor.decode(output[0], skip_special_tokens=True)
    return caption

def text_to_speech(text, output_path):
    tts = gTTS(text=text, lang="en")
    tts.save(output_path)

# Main Execution
video_path = 'videoplayback (online-video-cutter.com).mp4'
frame_folder = '/kaggle/working/frames'

# Step 1: Extract frames with movement detection
extract_frames(video_path, frame_folder)

# Step 2: Generate captions and convert to speech (SORT FILES NUMERICALLY)
frame_files = sorted(os.listdir(frame_folder), key=lambda x: int(x.split('_')[1].split('.')[0]))

for frame in frame_files:
    frame_path = os.path.join(frame_folder, frame)
    caption = generate_caption(frame_path)

    # Save audio file
    audio_path = os.path.join(frame_folder, f"{frame}.mp3")
    text_to_speech(caption, audio_path)
    
    print(f"{frame}: {caption} (Audio saved as {audio_path})")


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Extracted 298 frames with movement.
frame_0.jpg: a city street with people walking around it (Audio saved as /kaggle/working/frames/frame_0.jpg.mp3)
frame_1.jpg: a large cathedral with many windows and people walking around (Audio saved as /kaggle/working/frames/frame_1.jpg.mp3)
frame_2.jpg: a large cathedral with a clock tower in the middle (Audio saved as /kaggle/working/frames/frame_2.jpg.mp3)
frame_3.jpg: a large cathedral with many windows and a clock (Audio saved as /kaggle/working/frames/frame_3.jpg.mp3)
frame_4.jpg: a large cathedral with a clock tower in the middle (Audio saved as /kaggle/working/frames/frame_4.jpg.mp3)
frame_5.jpg: a large cathedral with a clock tower and a clock tower (Audio saved as /kaggle/working/frames/frame_5.jpg.mp3)
frame_6.jpg: a large cathedral with a clock tower and scatos (Audio saved as /kaggle/working/frames/frame_6.jpg.mp3)
frame_7.jpg: a large cathedral with many windows and a clock (Audio saved as /kaggle/working/frames/frame_7.jpg.mp3)
frame

In [8]:
import cv2
import os
import numpy as np
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from gtts import gTTS
from IPython.display import Audio, display
import tempfile

# Load the BLIP model for image captioning
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def extract_frames(video_path, output_folder, frame_rate=1):
    os.makedirs(output_folder, exist_ok=True)
    
    video = cv2.VideoCapture(video_path)
    fps = video.get(cv2.CAP_PROP_FPS)
    interval = int(fps * frame_rate)  # Capture every second

    count, saved = 0, 0
    prev_frame = None  # Store previous frame for movement detection

    while video.isOpened():
        ret, frame = video.read()
        if not ret:
            break

        if count % interval == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)  # Convert to grayscale for movement detection

            # Detect movement by comparing with previous frame
            if prev_frame is not None:
                diff = cv2.absdiff(prev_frame, gray)
                non_zero_count = np.count_nonzero(diff)
                movement_threshold = 5000  # Adjust threshold as needed
                
                if non_zero_count > movement_threshold:  # Movement detected
                    frame_path = os.path.join(output_folder, f"frame_{saved}.jpg")
                    cv2.imwrite(frame_path, frame)
                    saved += 1
            
            prev_frame = gray  # Update previous frame
        
        count += 1

    video.release()
    print(f"Extracted {saved} frames with movement.")

def generate_caption(image_path):
    image = Image.open(image_path).convert('RGB')
    inputs = processor(image, return_tensors="pt")
    output = model.generate(**inputs)
    caption = processor.decode(output[0], skip_special_tokens=True)
    return caption

def play_audio(text):
    """Convert text to speech and play it in Kaggle."""
    tts = gTTS(text=text, lang="en")
    
    with tempfile.NamedTemporaryFile(delete=True, suffix=".mp3") as fp:
        tts.save(fp.name)
        audio = Audio(fp.name, autoplay=True)
        display(audio)  # Ensure the audio is played in the Kaggle Notebook

# Main Execution
video_path = 'videoplayback (online-video-cutter.com).mp4'
frame_folder = '/kaggle/working/frames'

# Step 1: Extract frames with movement detection
extract_frames(video_path, frame_folder)

# Step 2: Generate captions and play audio
for frame in sorted(os.listdir(frame_folder)):
    frame_path = os.path.join(frame_folder, frame)
    caption = generate_caption(frame_path)

    print(f"{frame}: {caption}")
    
    # Play audio for the caption
    play_audio(caption)


Extracted 298 frames with movement.
frame_0.jpg: a city street with people walking around it


UnidentifiedImageError: cannot identify image file '/kaggle/working/frames/frame_0.jpg.mp3'

In [ ]:
print("hello")